In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("00 titanic.csv")

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Steps of Pipeline.

1) Imputing missing values in columns `Age` and `Embarked`.
2) Applying **One Hot Encoding** on columns `Sex` and `Embarked`.
3) Applying **Scaling**.
4) Applying **Feature Selection**.
5) Applying Decision tree model.

# Let's start.

First of all we'll drop some columns which are not required.

In [4]:
df.drop(columns = ['PassengerId', 'Name', 'Ticket', "Cabin"], inplace = True)

In [5]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [6]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [7]:
# Step 1: Splitting into train and test

x = df.iloc[:, 1:8]
y = df.drop(x, axis = 'columns')

print(x.head(3))

print()
print()
print()

print(y.head(3))

   Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0       3    male  22.0      1      0   7.2500        S
1       1  female  38.0      1      0  71.2833        C
2       3  female  26.0      0      0   7.9250        S



   Survived
0         0
1         1
2         1


In [8]:
# Train/ test split

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2)

In [9]:
print(x_train.head())

print()
print()
print()

print(y_train.head())

     Pclass   Sex   Age  SibSp  Parch    Fare Embarked
333       3  male  16.0      2      0  18.000        S
545       1  male  64.0      0      0  26.000        S
157       3  male  30.0      0      0   8.050        S
162       3  male  26.0      0      0   7.775        S
873       3  male  47.0      0      0   9.000        S



     Survived
333         0
545         0
157         0
162         0
873         0


In [10]:
# Step 2: Creating column transformers.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

In [11]:
# Creating transformer for imputing mmissing values

data_imputer = [
    ('impute_age', SimpleImputer(), [2]),
    ('impute_embarked', SimpleImputer(strategy = 'most_frequent'), [6])
]

# Here fpr columns we are passing index of columns instead of names bcz transformer returns numpy array and if passed name of column
# then this will create problem when passes these values to next transformers.

trans_imputer = ColumnTransformer(transformers = data_imputer, remainder = 'passthrough')

In [12]:
# Creating transformer for One Hot Encoder on "Sex" and "Embarked" column

data_ohe = [
    ('OHE_Sex_Embarked', OneHotEncoder(sparse = False, handle_unknown = 'ignore'), [1, 6])
]

trans_ohe = ColumnTransformer(transformers = data_ohe, remainder = 'passthrough')

In [36]:
# Creating transformer for Scaling.

data_scaling = [
    ('Scaler', MinMaxScaler(), slice(0, 10))
]

# slice(0,10) means apply scaler on all the rows from 0 to 10. 
# We have 7 columns then why are we mentioning 0 to 10?
# It's bcz after applying OneHotEncoder on 'Sex' and 'Embarked' column, both these columns are removed
# and 2 and 3 new columns got created from 'Age' and 'Embarked' respectively. Therefore, 7-2+2+3 = 10

trans_scaling = ColumnTransformer(transformers = data_scaling)

In [37]:
# Feature Selection

from sklearn.feature_selection import SelectKBest, chi2

In [38]:
trans_feature_selection = SelectKBest(score_func = chi2, k = 8)

In [39]:
# Train the model

from sklearn.tree import DecisionTreeClassifier

trans_dt = DecisionTreeClassifier()

# Create Pipeline

In [40]:
from sklearn.pipeline import Pipeline, make_pipeline

In [41]:
pipe = Pipeline([
    ('trans1', trans_imputer),
    ('trans2', trans_ohe),
    ('trans3', trans_scaling),
    ('trans4', trans_feature_selection),
    ('trans_5', trans_dt)
])

# Pipeline VS make_pipeline()

* The only difference betweenn Pipeline and make_pipeline() is in make_pipeline() we don't need to pass the names of the transformers.
* The same goes with ColumnTransformer VS make_column_transformer()

In [42]:
pipe2 = make_pipeline(trans_imputer, trans_ohe, trans_scaling, trans_feature_selection, trans_dt)

# Train the model on pipeline named 'pipe'

In [43]:
pipe.fit(x_train, y_train)

# If we are not training our nodel through pipeline, then in that case we would use 'fit_transform' instead of 'fit'.

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Pipeline(steps=[('trans1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trans2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('OHE_Sex_Embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse=False),
                                                  [1, 6])])),
                ('trans3',
                 ColumnTransformer(transformers=[('Scaler', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trans4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000002978D37EC00>)),
                ('trans_5', DecisionTreeClassifier())])

# Extract some information from the pipeline.

In [58]:
pipe.named_steps

{'trans1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trans2': ColumnTransformer(remainder='passthrough',
                   transformers=[('OHE_Sex_Embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse=False),
                                  [1, 6])]),
 'trans3': ColumnTransformer(transformers=[('Scaler', MinMaxScaler(), slice(0, 10, None))]),
 'trans4': SelectKBest(k=8, score_func=<function chi2 at 0x000002978D37EC00>),
 'trans_5': DecisionTreeClassifier()}

In [45]:
pipe.named_steps['trans1']

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [47]:
pipe.named_steps['trans1'].transformers_

[('impute_age', SimpleImputer(), [2]),
 ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
 ('remainder', 'passthrough', [0, 1, 3, 4, 5])]

In [48]:
pipe.named_steps['trans1'].transformers_[0]

('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])

In [49]:
pipe.named_steps['trans1'].transformers_[0][1]

SimpleImputer()

In [51]:
pipe.named_steps['trans1'].transformers_[0][1].statistics_

# This is the mean of the age column. Bcz we applied Simple Imputer with strategy = 'mean'

array([29.50572183])

In [52]:
pipe.named_steps['trans1'].transformers_[1]

('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])

In [53]:
pipe.named_steps['trans1'].transformers_[1][1]

SimpleImputer(strategy='most_frequent')

In [56]:
pipe.named_steps['trans1'].transformers_[1][1].statistics_

# This shows the most_frequent value of 'Embarked' column bcz we applied strategy = 'most_frequent'
# to 'Embarked' column.

array(['S'], dtype=object)

# Make prediction

In [61]:
pred = pipe.predict(x_test)

print(pred)

[0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0
 1 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 0 0 1 1 0 0 1 0 0 0 0 0 0
 0 0 1 0 0 1 1 0 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 1 0 1 0 1 0 1 1 0
 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]


In [60]:
# Getting accuracy

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

print(accuracy)

0.6480446927374302


# Cross validation using Pipeline.

`Cross validation` has one parameter named **cv** whics accepts integer value, from many other parameters.
* For eg. If we set **cv = 5**, then it will perform the train/ test split 5 times and train the model on every single batch of train/ test split.
* It has one more parameter named **scoring**. If assigned `accuracy` to along with `mean` then for accuracy it takes average of the accuracies of all the 5 batches.

**from sklearn.model_selection import cross_val_score**
# Some commonly used parameters of cross_val_score are:
* `estimator`: Takes the object of the Pipeline.
*  `X`: Takes x_train.
* `y`: Takes y_train.
* `cv`: Takes no.of cross validations we wanna perform.
* `scoring`: Takes the method to count the accuracy. Mostly **accuracy** is used.* 
'v_measure_score': V-measure

In [62]:
from sklearn.model_selection import cross_val_score

In [69]:
cross_val_score(estimator = pipe, X = x_train, y = y_train, cv = 5, scoring = 'precision')

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocess

array([0.6       , 0.625     , 0.5       , 0.57692308, 0.52      ])

In [66]:
cross_val_score(estimator = pipe, X = x_train, y = y_train, cv = 5, scoring = 'accuracy').mean()

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocess

0.6333891460652025

# Grid Search using Pipeline

In [76]:
# gridsearchcv

params = {
    'trans_5__max_depth': [1,2,3,4,5,None]
}

In [77]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipe, params, cv = 5, scoring = 'accuracy')
grid.fit(x_train, y_train)

C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocess

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trans1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trans2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('OHE_Sex_Embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse=False),
                                                                         [1,
                                                                          6])])),
                                       ('trans3',
                                        ColumnTransformer(transformers=[('Scaler',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trans4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x000002978D37EC00>)),
                                       ('trans_5', DecisionTreeClassifier())]),
             param_grid={'trans_5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [78]:
grid.best_score_

0.6333891460652025

In [79]:

grid.best_params_

{'trans_5__max_depth': 2}

# Exporting the Pipeline

In [80]:
import pickle

pickle.dump(pipe, open('pipe.pkl', 'wb'))

# Next: 04 Predicting using Pipeline.